# Train your first reducer selector

Choose **Run → Run All Cells**. This notebook uses recorded teacher data: no simulation or teacher collection is needed.

Your network scores four sound native reducer actions: `girard`, `scott`, `pca`, `combastel`. Lower scores rank first. The native execution layer handles bounds and fallbacks. Training uses the existing pairwise objective, rather than predicting calibrated costs.

The starter contains budget-40 decision states from training seeds 0–3 and validation seeds 20–21. Validation is for development, not a fresh generalization claim.

In [ ]:
from pzr import student
import pandas as pd
import torch

torch.set_num_threads(1)
task = student.load_task("robot_arm", bundle="starter", budget=40)

In [ ]:
pd.DataFrame({"split": ["train", "validation"],
              "states": [task.training.num_samples, task.validation.num_samples]})

## Train and inspect validation results
Twenty epochs keep the first run small. The returned policy uses the best validation checkpoint.

In [ ]:
run = student.train_selector(task, config=student.TrainingConfig(epochs=20, seed=42))
pd.Series(student.evaluate_selector(run.policy, task.validation))

## Compare with the frozen reference
The frozen Clean148 model used 148 training trajectories. This is a reference, not a controlled comparison with the four-trajectory starter model. Notebook 01 compares two networks trained on identical data.

In [ ]:
pd.DataFrame({
    "starter MLP": student.evaluate_selector(run.policy, task.validation),
    "frozen Clean148 (more training data)": student.evaluate_selector(task.frozen_baseline(), task.validation),
})

## Run in the native monitor
This short prefix checks integration. Prefix losses are explicitly labelled; remove `max_events` for full 500-event validation trajectories. Exact references may take longer than neural training.

In [ ]:
student.evaluate_monitor(run.policy, task.validation_traces, max_events=20,
                         reference_cache="workspace/references")

## Save and reload
Each execution uses a new directory, so rerunning the notebook preserves earlier experiments.

In [ ]:
from pathlib import Path
from uuid import uuid4
output = Path("workspace") / ("first-selector-" + uuid4().hex[:8])
run.save(output)
reloaded = student.load_selector(output)
print(output)
assert student.evaluate_selector(reloaded, task.validation) == student.evaluate_selector(run.policy, task.validation)